# Trabalho 1 — TDD e propriedades em pipelines de ML
## Testes Automatizados — IEC PUC Minas

**Aluno:** Andre Cardoso de Oliveira

Notebook no mesmo formato das aulas (Colab + `ipytest` + `hypothesis`).

O enunciado pede 4 elementos. Este notebook cobre os quatro:

1. **TDD** (Red → Green → Refactor) — relato visível nas células, como na Aula 2.
2. **Teste de propriedade** — invariante com Hypothesis, para qualquer entrada válida.
3. **Versão bugada** — a mesma propriedade falha contra uma função propositalmente errada.
4. **Decisões de design** — seção 4, no fim.

## Função escolhida

`most_confident_class(scores)` — **seleção de classe** depois de um `softmax`.

Recebe uma lista de scores (um por classe) e devolve o **índice** do maior valor (`argmax`). É o passo do pipeline que transforma a saída do modelo na classe prevista.

Não é o `clip_score` da aula ao vivo: a técnica de TDD é a mesma, o problema é outro.

**Decisão de design (empate):** se duas classes empatam no máximo, devolvemos o **menor índice**. Motivo: `argmax` estável, previsível e fácil de testar. Fica documentada e coberta por teste no Green.

In [ ]:
!pip install -q "ipytest==0.14.*" "hypothesis==6.164.*"

import ipytest
import pytest
ipytest.autoconfig()

---
## 1. TDD — `most_confident_class`

Ciclo da Aula 2: **vermelho → verde → refatoração**. Um teste por vez.

### 🔴 Passo 1 — Red

O teste é escrito **antes** da função existir. Deve falhar (`most_confident_class` ainda não existe). Isso não é erro: é o ponto de partida do TDD.

Caso: scores `[0.1, 0.7, 0.2]` → classe de índice `1` (a mais confiante).

In [ ]:
%%ipytest

# garante o estado inicial do TDD mesmo se você já rodou células mais à frente
try:
    del most_confident_class
except NameError:
    pass


def test_most_confident_class_returns_index_of_max():
    assert most_confident_class([0.1, 0.7, 0.2]) == 1

### Relato Red

Rode a célula acima. O relatório do `pytest` deve mostrar **FAILED** / `NameError: name 'most_confident_class' is not defined`.

**Não implemente a função ainda.** Este estado (teste existe, código ainda não) é o Red.

A seção 2 (propriedade) está abaixo. Rode essa célula só depois do Green.

### 🟢 Passo 2 — Green (o mínimo possível)

Implementação propositalmente incompleta (*fake it*), como no `clip_score` da Aula 2: só o suficiente para passar **este** teste. `return 1` funciona para `[0.1, 0.7, 0.2]`, mas ainda não é a função de verdade.

In [ ]:
%%ipytest

def most_confident_class(scores):
    return 1


def test_most_confident_class_returns_index_of_max():
    assert most_confident_class([0.1, 0.7, 0.2]) == 1

### 🔴🟢 Um novo teste força a implementação real

Com o `return 1`, o primeiro teste continua passando. O novo caso tem o máximo no índice `0` — a implementação falsa não sobrevive.

In [ ]:
%%ipytest

def most_confident_class(scores):
    return 1


def test_most_confident_class_returns_index_of_max():
    assert most_confident_class([0.1, 0.7, 0.2]) == 1


def test_most_confident_class_when_max_is_first():
    assert most_confident_class([0.9, 0.05, 0.05]) == 0

### 🟢 Implementação real

Agora a lógica de verdade: devolver o índice do maior score. Em empate, `list.index` pega a **primeira** ocorrência — o menor índice, como decidimos.

In [ ]:
%%ipytest

def most_confident_class(scores):
    return scores.index(max(scores))


def test_most_confident_class_returns_index_of_max():
    assert most_confident_class([0.1, 0.7, 0.2]) == 1


def test_most_confident_class_when_max_is_first():
    assert most_confident_class([0.9, 0.05, 0.05]) == 0


def test_most_confident_class_handles_tie():
    assert most_confident_class([0.5, 0.5, 0.1]) == 0

### Relato Green

1. Célula *fake it* (`return 1`): o teste Red passa — **1 passed**. Ainda não é a função real.
2. Célula com o segundo caso (`[0.9, 0.05, 0.05] → 0`): o fake quebra — **FAILED**.
3. Célula da implementação real (`scores.index(max(scores))`): os três testes passam, inclusive o empate.

Depois disso, a célula de **propriedade** (abaixo) já pode rodar: a função existe, e o Hypothesis vai gerar muitas listas.

### 🔵 Passo 3 — Refactor

Só refatoramos com os testes verdes. A lógica não muda: extraímos o máximo para um nome mais claro, documentamos o contrato (incluindo o empate) e tipamos a assinatura. Os três testes do Green têm que continuar passando.

In [ ]:
%%ipytest

def most_confident_class(scores: list[float]) -> int:
    """Devolve o índice da classe com maior score (argmax).

    Em empate, permanece o menor índice (primeira ocorrência).
    """
    highest_score = max(scores)
    return scores.index(highest_score)


def test_most_confident_class_returns_index_of_max():
    assert most_confident_class([0.1, 0.7, 0.2]) == 1


def test_most_confident_class_when_max_is_first():
    assert most_confident_class([0.9, 0.05, 0.05]) == 0


def test_most_confident_class_handles_tie():
    assert most_confident_class([0.5, 0.5, 0.1]) == 0

### Relato Refactor

A célula acima deve mostrar **3 passed**. Se algum teste falhasse, o refactor teria mudado o comportamento — e aí voltaríamos atrás.

O que mudou (só clareza):

- docstring com o contrato e a decisão de empate
- `highest_score` no lugar de `max(scores)` inline
- tipos `list[float] -> int`

O que **não** mudou: para as mesmas entradas, as mesmas saídas.

---
## 2. Teste de propriedade (invariante)

O teste Red acima é **um exemplo pontual**: para `[0.1, 0.7, 0.2]`, esperamos `1`.

Um teste de propriedade troca essa pergunta por outra: **para qualquer entrada válida, esta afirmação é sempre verdadeira?** O `hypothesis` gera dezenas de listas sozinho e procura um contraexemplo (e reduz ao caso mais simples — *shrinking*).

### Invariante escolhida

> O índice devolvido **sempre aponta para um valor máximo** da lista.

Em código: `scores[most_confident_class(scores)] == max(scores)`.

Isso vale no caso pontual `[0.1, 0.7, 0.2]`, no empate `[0.5, 0.5, 0.1]` (os dois primeiros são máximos; devolver `0` continua correto) e em qualquer lote finito não vazio.

**Domínio válido** (igual à Aula 2 no `clip_score`): floats finitos, sem `NaN` e sem `inf` (`allow_nan=False`, `allow_infinity=False`). Lista com pelo menos 1 elemento (`min_size=1`) — `argmax` de lista vazia não é entrada válida.

**Quando rodar:** só depois do Green. Sem a função, este teste também falha — e aí não estamos testando a propriedade, só a ausência do código.

In [ ]:
%%ipytest

from hypothesis import given, strategies as st

# Rode depois do Green. A função precisa existir.


@given(st.lists(st.floats(allow_nan=False, allow_infinity=False, width=32), min_size=1))
def test_most_confident_class_points_to_max_value(scores):
    idx = most_confident_class(scores)
    assert scores[idx] == max(scores)

---
## 3. A propriedade pega um bug real

Como na Aula 2 com o `clip_score_buggy`: a mesma invariante, agora contra uma versão **errada**.

O bug é plausível em produção: o classificador **sempre devolve a classe 0** (esqueceu o `argmax` e fixou o índice). O teste de exemplo `[0.1, 0.7, 0.2] → 1` também falharia, mas o ponto aqui é outro: a propriedade não precisa desse exemplo. Ela gera entradas até achar um caso em que `scores[0] != max(scores)` e reduz (*shrinking*) ao contraexemplo mais simples.

Rode a célula. Esperado: **FAILED**, com um contraexemplo do Hypothesis — não `NameError`.

In [ ]:
%%ipytest

from hypothesis import given, strategies as st


def most_confident_class_buggy(scores):
    return 0


@given(st.lists(st.floats(allow_nan=False, allow_infinity=False, width=32), min_size=1))
def test_buggy_version_points_to_max_value(scores):
    idx = most_confident_class_buggy(scores)
    assert scores[idx] == max(scores)

### Relato — o teste teria pego o bug

A célula acima deve falhar. O Hypothesis encontra uma lista em que o máximo **não** está no índice 0 (por exemplo algo como `[0.0, 1.0]`) e mostra esse contraexemplo no relatório.

Isso é a prova pedida: a propriedade, sozinha, detecta a função bugada. Em produção, um retreino ou um `return 0` esquecido não passaria nessa suíte.

---
## 4. Decisões de design

Aplicáveis a `most_confident_class`. O que o enunciado cita e **não** cabe aqui fica explícito.

| Tópico | Decisão | Por quê |
|---|---|---|
| **Empate** | Menor índice (primeira ocorrência). `list.index(max(scores))` já faz isso. | `argmax` estável; coberto pelo teste Green `[0.5, 0.5, 0.1] → 0`. |
| **Lista vazia** | Entrada inválida. Não é domínio da função (`min_size=1` no Hypothesis). `max([])` levantaria `ValueError`. | Não existe classe prevista sem scores. |
| **NaN** | Score inválido. Fora do domínio da propriedade (`allow_nan=False`). | `max` com `NaN` é instável; um NaN não é confiança utilizável. Validar entrada é etapa **antes** do `argmax` (lição da Aula 2 com `clip_score(NaN)`). |
| **inf / -inf** | Score inválido. Fora do domínio (`allow_infinity=False`). | Infinito costuma ser overflow, não probabilidade/confiança. |
| **Divisão por zero** | Não se aplica. A função só compara e devolve índice. | — |
| **Tipos** | Lista de `float` (saída de `softmax`). `bool` e string numérica não são score. | Validar ≠ converter; se chegou `True` ou `"0.5"`, o erro é de quem chamou. |
| **Entrada inválida** | Esta função não “conserta” nem devolve sentinela. O contrato é: scores finitos, lista não vazia. | A propriedade só garante o que o domínio declara. Quem serve o modelo valida antes. |

A propriedade testada (`scores[idx] == max(scores)`) **não** distingue empate no primeiro ou no último máximo — os dois são máximos. A escolha do menor índice é decisão extra, documentada e coberta por teste de exemplo, não pela invariante.